# Estudo de caso: ANOVA com o conjunto PlantGrowth

Este notebook aplica a ANOVA de uma via ao conjunto clássico `PlantGrowth`. O objetivo é verificar se o peso médio das plantas difere entre um grupo-controle (`ctrl`) e dois tratamentos (`trt1` e `trt2`).

A análise segue o roteiro de `GCC1625_05_anova.tex`: descrever os grupos, verificar as condições de aplicabilidade, aplicar a ANOVA com `f_oneway` e, se apropriado, usar o teste post-hoc de Tukey.


## 1. Preparação

A variável independente é `group`, com três níveis; a variável dependente é `weight`, quantitativa. As hipóteses são:

$$H_0: \mu_{ctrl}=\mu_{trt1}=\mu_{trt2}$$

$$H_a: \text{ao menos uma média populacional difere.}$$


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import f_oneway, levene, probplot, shapiro
from statsmodels.stats.multicomp import pairwise_tukeyhsd

ALPHA = 0.05
sns.set_theme(style='whitegrid', palette='Set2')


In [ ]:
# Execute o notebook a partir da pasta notebooks/.
arquivo = Path('../data/anova/PlantGrowth.csv')
dados = pd.read_csv(arquivo)

dados['group'] = pd.Categorical(
    dados['group'],
    categories=['ctrl', 'trt1', 'trt2'],
    ordered=True,
)
dados.head()


## 2. Descrição e visualização inicial

Antes do teste, observamos tamanho, média e dispersão de cada grupo. O boxplot mostra mediana e quartis; os pontos mostram as observações individuais. No gráfico de pontos, o deslocamento horizontal é apenas *jitter*: evita que valores próximos fiquem sobrepostos e não representa uma segunda variável.


In [ ]:
resumo = dados.groupby('group', observed=False)['weight'].agg(
    n='count', media='mean', desvio_padrao='std', minimo='min', maximo='max'
)
resumo.round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(
    data=dados, x='group', y='weight', hue='group',
    palette='Set2', legend=False, ax=axes[0],
)
sns.stripplot(
    data=dados, x='group', y='weight', color='black',
    jitter=0.12, alpha=0.7, ax=axes[0],
)
axes[0].set(title='Peso por grupo', xlabel='Grupo', ylabel='Peso')

for grupo, cor in zip(dados['group'].cat.categories, sns.color_palette('Set2', 3)):
    axes[1].hist(
        dados.loc[dados['group'] == grupo, 'weight'],
        bins=5, alpha=0.5, label=grupo, color=cor,
    )
axes[1].set(title='Histogramas por grupo', xlabel='Peso', ylabel='Frequência')
axes[1].legend(title='Grupo')

plt.tight_layout()


**Leitura dos gráficos.** Os grupos têm o mesmo tamanho amostral e dispersões visualmente semelhantes. O grupo `trt2` parece ter média maior, enquanto `trt1` parece ter média menor que o controle. Essa inspeção é descritiva: a decisão inferencial será feita pela ANOVA, após avaliar as condições.


## 3. Condições de aplicabilidade

A independência depende do desenho do experimento e não pode ser verificada pelo código. Aqui, supomos que cada planta foi medida uma única vez e que as unidades foram alocadas aos grupos de modo independente.

Para a homocedasticidade, aplicamos Levene. Sua hipótese nula afirma igualdade das variâncias populacionais.


In [ ]:
grupos = {
    nome: dados.loc[dados['group'] == nome, 'weight']
    for nome in dados['group'].cat.categories
}

estatistica_levene, p_levene = levene(*grupos.values(), center='mean')
print(f'Estatística de Levene = {estatistica_levene:.3f}')
print(f'p-valor = {p_levene:.4f}')


**Leitura do teste de Levene.** Compare o p-valor com $$\alpha = 0{,}05.$$ Se ele for maior que $$\alpha,$$ não rejeitamos a hipótese de variâncias iguais: não há evidência suficiente de heterogeneidade das variâncias. Isso não prova que as variâncias sejam idênticas; deve ser avaliado junto com os gráficos e os desvios-padrão apresentados acima.


Para examinar a normalidade, usamos gráficos Q--Q e o teste de Shapiro--Wilk em cada grupo. Em cada teste, a hipótese nula é que os dados do grupo provêm de uma distribuição normal. Com apenas dez observações por grupo, ambos são evidências auxiliares e devem ser lidos com cautela.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

resultados_shapiro = []
for ax, (nome, valores) in zip(axes, grupos.items()):
    estatistica, p_valor = shapiro(valores)
    resultados_shapiro.append(
        {'grupo': nome, 'estatística W': estatistica, 'p-valor': p_valor}
    )
    probplot(valores, dist='norm', plot=ax)
    ax.set_title(f'Q--Q: {nome}')

plt.tight_layout()

pd.DataFrame(resultados_shapiro).set_index('grupo').round(4)


**Leitura de Shapiro--Wilk e dos Q--Q plots.** Para cada grupo, um p-valor maior que $$\alpha$$ significa que o teste não encontrou evidência suficiente contra a normalidade. Pontos aproximadamente alinhados à reta no gráfico Q--Q contam a mesma história visual. Nem um p-valor alto prova normalidade, nem um p-valor baixo isolado dispensa examinar o gráfico e o contexto do estudo.


## 4. ANOVA de uma via

Com as condições consideradas razoáveis, aplicamos `f_oneway`. A estatística $$F$$ compara a variabilidade entre as médias dos grupos com a variabilidade dentro deles. Um p-valor pequeno indica evidência contra a igualdade de todas as médias, mas ainda não diz quais pares diferem.


In [ ]:
f_stat, p_valor = f_oneway(*grupos.values())

print(f'F = {f_stat:.4f}')
print(f'p-valor = {p_valor:.4f}')
if p_valor <= ALPHA:
    print('Decisão: rejeitamos H0; há evidência de diferença entre pelo menos duas médias.')
else:
    print('Decisão: não rejeitamos H0; não há evidência suficiente de diferença entre as médias.')


**Leitura da ANOVA.** Compare o p-valor com $$\alpha = 0{,}05.$$ Quando $$p \leq \alpha,$$ rejeitamos $$H_0$$ e concluímos que há evidência de que pelo menos um par de médias populacionais difere. Essa conclusão é global: ela não permite afirmar, sozinha, quais tratamentos diferem entre si. Para isso, usamos Tukey.


## 5. Comparações post-hoc de Tukey

Como a ANOVA é significativa neste conjunto, aplicamos Tukey HSD a todos os pares. O método ajusta as comparações múltiplas. A coluna `reject` indica se a diferença do par é significativa ao nível de 5%; intervalos que não contêm zero conduzem à mesma decisão.


In [ ]:
tukey = pairwise_tukeyhsd(
    endog=dados['weight'],
    groups=dados['group'],
    alpha=ALPHA,
)

print(tukey.summary())


In [ ]:
tukey.plot_simultaneous()
plt.title('Intervalos simultâneos de Tukey para PlantGrowth')
plt.xlabel('Peso médio')
plt.show()


**Leitura do teste de Tukey.** Examine cada linha da tabela. Para um par com `reject=True`, há evidência de diferença entre as respectivas médias após o ajuste de Tukey; com `reject=False`, os dados não fornecem evidência suficiente para distinguir aquelas médias. No conjunto PlantGrowth, a comparação entre `trt1` e `trt2` é a que apresenta evidência de diferença; as conclusões para os demais pares devem ser lidas diretamente da tabela produzida na sua execução.


## Conclusão

Este estudo de caso ilustra o fluxo completo da ANOVA de uma via:

1. descrever a variável resposta em cada grupo;
2. avaliar independência pelo desenho, homocedasticidade por Levene e gráficos, e normalidade por Q--Q e Shapiro--Wilk;
3. aplicar `f_oneway` para o teste global;
4. aplicar Tukey apenas quando houver rejeição de $$H_0$$;
5. interpretar o resultado no contexto do experimento, sem confundir associação estatística com explicação causal.
